In [ ]:
import ROOT
import numpy as np
import pandas as pd
 
ROOT.gStyle.SetOptFit(1111)    # aggiunge il label con chi2/ndf e probabilita e parametri se metti 0000 toglie i dati

#fit 
def Sellmeier(x, par):
    l = x[0]
    A = par[0]
    B = par[1]
    return np.sqrt(1 + ( (A*l**2)/(l**2 - B**2) ))

def modellopy(l, A, B):         
    return np.sqrt(1 + (A*l**2)/(l**2 - B**2))


def Sell_deriv(x, par):
    l = x
    dl = 1e-6
    return ((modellopy(l+dl/2, par[0], par[1])-modellopy(l-dl/2, par[0], par[1]))/dl)
    


#leggere file excel 
xlsx ="esperienza3.xlsx"
df = pd.read_excel("esperienza3.xlsx", sheet_name= "Dati")
df = df.dropna(subset=["lambda", "n'", "err_lambda", "err_n"])
df = df.to_numpy(dtype = 'float64')

x = df[:, 0]     
y = df[:, 9] 

err_x = df[:, 1] 
err_y = df[:, 11]


#minimizzazzione con ROOT
SellFit = ROOT.TF1('model', Sellmeier, min(x), max(x), 2)
SellFit.SetParameters(1.0, 100.0)     
SellFit.SetParNames("A", "B")

gr1 = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr1.SetTitle("Dati sperimentali e fit; #lambda [nm]; n")
gr1.SetMarkerStyle(20)
gr1.SetMarkerSize(0.7)                    
gr1.SetMarkerColor(ROOT.kAzure + 2)
gr1.SetLineColor(ROOT.kAzure + 3)        
gr1.SetLineWidth(1)


fit_res = gr1.Fit(SellFit, "SR")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

par = SellFit.GetParameters()

errors = SellFit.GetParErrors()

chi2 = fit_res.Chi2()
ndf = fit_res.Ndf()
chi2_sell, ndf_sell = chi2, ndf      

err_yeff = np.sqrt(err_y**2 + ((Sell_deriv(x, par)) * err_x)**2) #errore effettivo

print("Parametri stimati:")
print(f"A = {par[0]} ± {errors[0]}")
print(f"B = {par[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}, ndf = {ndf}")

c1 = ROOT.TCanvas("c1", "Fit con ROOT", 800, 600)
c1.SetGrid()

gr1.Draw("AP")


leg1 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg1.AddEntry(gr1, "Dati", "pe")
leg1.AddEntry(SellFit, "Fit", "l")
leg1.Draw()

c1.Draw()
c1.SaveAs("Immagini/fit_root.png")

Fit status: 0
Parametri stimati:
A = 1.2618987276322813 ± 0.0014526047556983042
B = 101.025532328039 ± 1.990909569527899
Chi2 = 2.247844977348605, ndf = 12
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      2.24784
NDf                       =           12
Edm                       =  3.87288e-08
NCalls                    =           95
A                         =       1.2619   +/-   0.0014526   
B                         =      101.026   +/-   1.99091     


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file Immagini/fit_root.png has been created


In [165]:
if np.abs(par[0]) < 3*errors[0]:
        print("A è compatibile con 0")
else :
        print("A NON è compatibile con 0")

print("---")

if np.abs(par[1]) < 3*errors[1]:
    print("B è compatibile con 0")
else :
    print("B NON è compatibile con 0")
 
 


A NON è compatibile con 0
---
B NON è compatibile con 0


In [166]:
def lineare(x, par):
    return par[0] + par[1] * x[0]

LinFit = ROOT.TF1('lineare', lineare, min(x), max(x), 2)
LinFit.SetParameters(1.5, 0.0)         
LinFit.SetParNames("A", "B")
LinFit.SetLineColor(ROOT.kGreen + 2)
LinFit.SetLineStyle(2)

gr_lin = ROOT.TGraphErrors(len(x), x, y, err_x, err_y)
gr_lin.SetTitle("Confronto tra i modelli; #lambda [nm]; n")
gr_lin.SetMarkerStyle(20)
gr_lin.SetMarkerSize(0.7)
gr_lin.SetMarkerColor(ROOT.kAzure + 2)
gr_lin.SetLineColor(ROOT.kAzure + 3)

res_lin = gr_lin.Fit(LinFit, "SR")
print(f"Fit status: {res_lin.Status()}")

par_lin = LinFit.GetParameters()
err_lin = LinFit.GetParErrors()
chi2_lin = res_lin.Chi2()
ndf_lin = res_lin.Ndf()

print("Parametri stimati (fit lineare):")
print(f"A = {par_lin[0]} ± {err_lin[0]}")
print(f"B = {par_lin[1]} ± {err_lin[1]}")

print()
print(f"Sellmeier: chi2/ndf = {chi2_sell:.2f}/{ndf_sell} ")
print(f"Lineare:   chi2/ndf = {chi2_lin:.2f}/{ndf_lin} ")
print(f"Delta chi2 (lineare - Sellmeier) = {chi2_lin - chi2_sell:.2f}")




c_lin = ROOT.TCanvas("c_lin", "Fit lineare", 800, 600)
c_lin.SetGrid()

gr_lin.SetTitle("Fit lineare; #lambda [nm]; n")
gr_lin.Draw("AP")
LinFit.Draw("SAME")

leg_lin = ROOT.TLegend(0.60, 0.75, 0.89, 0.89)
leg_lin.SetTextSize(0.03)  
leg_lin.AddEntry(gr_lin, "Dati", "pe")
leg_lin.AddEntry(LinFit, "Fit lineare", "l")
leg_lin.Draw()

c_lin.Draw()
c_lin.SaveAs("Immagini/fit_lineare.png")


c_comp = ROOT.TCanvas("c_comp", "Confronto modelli", 800, 600)
c_comp.SetGrid()

gr_lin.SetTitle("Confronto tra i modelli; #lambda [nm]; n")
gr_lin.Draw("AP")
LinFit.Draw("SAME")
SellFit.Draw("SAME")

leg_comp = ROOT.TLegend(0.60, 0.70, 0.89, 0.89)
leg_comp.SetTextSize(0.03)  
leg_comp.AddEntry(gr_lin, "Dati", "pe")
leg_comp.AddEntry(LinFit, "Fit lineare", "l")
leg_comp.AddEntry(SellFit, "Fit Sellmeier", "l")
leg_comp.Draw()

c_comp.Draw()
c_comp.SaveAs("Immagini/fit_confronto.png")



Fit status: 0
Parametri stimati (fit lineare):
A = -14103.156207167869 ± 1936.1666983245157
B = 24.014154977652744 ± 3.2964308046519486

Sellmeier: chi2/ndf = 2.25/12 
Lineare:   chi2/ndf = 839.04/12 
Delta chi2 (lineare - Sellmeier) = 836.79
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      839.039
NDf                       =           12
Edm                       =  1.14023e-05
NCalls                    =          253
A                         =     -14103.2   +/-   1936.17     
B                         =      24.0142   +/-   3.29643     


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_lin
Info in <TCanvas::Print>: png file Immagini/fit_lineare.png has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_comp
Info in <TCanvas::Print>: png file Immagini/fit_confronto.png has been created


In [167]:
y_attesi = np.array([SellFit.Eval(x_val) for x_val in x])
residuals_norm = (y - y_attesi) / err_yeff



gaussian_fit = ROOT.TF1('gaussian', 'gaus', min(residuals_norm), max(residuals_norm), 3)
gaussian_fit.SetParameters(1, 0, 1)
gaussian_fit.SetParNames("const", "mu", "sigma")
gaussian_fit.SetLineColor(ROOT.kRed + 1)
gaussian_fit.SetLineWidth(3)

hist = ROOT.TH1F("hist", "Istogramma dei dati; x; y", 7, min(residuals_norm), max(residuals_norm))
for val in residuals_norm:
    hist.Fill(val)
    
hist.SetLineColor(ROOT.kAzure + 3)
hist.SetLineWidth(2)
hist.SetFillColorAlpha(ROOT.kAzure + 1, 0.55)   
hist.GetXaxis().SetTitle("residuo normalizzato")
hist.GetYaxis().SetTitle("conteggi")
hist.SetTitle("Istogramma residui normalizzati")

r = hist.Fit(gaussian_fit, "S")

fitStatus = r.Status()
print(f"Fit status: {fitStatus}")

parameters = gaussian_fit.GetParameters()
errors = gaussian_fit.GetParErrors()

chi2 = gaussian_fit.GetChisquare()

print("Parametri stimati:")
print(f"const = {parameters[0]} ± {errors[0]}")
print(f"#mu = {parameters[1]} ± {errors[1]}")
print(f"#sigma = {parameters[2]} ± {errors[2]}")
print(f"Chi2 = {chi2}")


c2 = ROOT.TCanvas("c2", "Istogramma dei dati", 800, 600)
c2.SetGrid()

hist.Draw()

leg2 = ROOT.TLegend(0, 0.30, 0.15, 0.39)
leg2.AddEntry(hist, "Residui", "f")
leg2.AddEntry(gaussian_fit, "Fit gaussiano", "l")
leg2.Draw()


c2.Draw()
c2.SaveAs("Immagini/hist.png")


Warning in <TROOT::Append>: Replacing existing TH1: hist (Potential memory leak).


Fit status: 0
Parametri stimati:
const = 8.118598931043588 ± 2.9011996534965054
#mu = -0.22146188825860497 ± 0.03842792662075052
#sigma = 0.12217019654653026 ± 0.025294978326439753
Chi2 = 1.992032847965843
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      1.99203
NDf                       =            2
Edm                       =  4.30709e-06
NCalls                    =           94
const                     =       8.1186   +/-   2.9012      
mu                        =    -0.221462   +/-   0.0384279   
sigma                     =      0.12217   +/-   0.025295     	 (limited)


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c2
Info in <TCanvas::Print>: png file Immagini/hist.png has been created


In [168]:
if np.abs(parameters[0] - 1) < 3*errors[0]:
    print("const è compatibile con 1")
else :
    print("const NON è compatibile con 1")

print("---")

if np.abs(parameters[1]) < 3*errors[1]:
    print("mu è compatibile con 0")
else :
    print("mu NON è compatibile con 0")

print("---")

if np.abs(parameters[2] - 1) < 3*errors[2]:
    print("sigma è compatibile con 1")
else :
    print("sigma NON è compatibile con 1")

const è compatibile con 1
---
mu NON è compatibile con 0
---
sigma NON è compatibile con 1


In [169]:
def line(x, par):
    x = x[0]
    m = par[0]
    q = par[1]
    return m * x + q

residuals = y - y_attesi

line_fit = ROOT.TF1('line', line, min(x), max(x), 2)
line_fit.SetParameters(1, 0)
line_fit.SetParNames("m", "q")
line_fit.SetLineColor(ROOT.kRed + 1)
line_fit.SetLineWidth(2)

gr2 = ROOT.TGraphErrors(len(x), x, residuals, np.zeros_like(residuals), err_yeff)
gr2.SetTitle("Residui in funzione di #lambda; #lambda [nm]; n - n_{att}")
gr2.SetMarkerStyle(20)
gr2.SetMarkerSize(0.7)
gr2.SetMarkerColor(ROOT.kAzure + 2)
gr2.SetLineColor(ROOT.kAzure + 3)

fit_res = gr2.Fit(line_fit, "S")

fitStatus = fit_res.Status()
print(f"Fit status: {fitStatus}")

parameters = line_fit.GetParameters()
errors = line_fit.GetParErrors()

chi2 = fit_res.Chi2()

print("Parametri stimati:")
print(f"m = {parameters[0]} ± {errors[0]}")
print(f"q = {parameters[1]} ± {errors[1]}")
print(f"Chi2 = {chi2}")

c3 = ROOT.TCanvas("c3", "Fit con ROOT", 800, 600)
c3.SetGrid()

gr2.Draw("AP")

leg3 = ROOT.TLegend(0.17, 0.80, 0.45, 0.89)
leg3.AddEntry(gr2, "Residui", "pe")
leg3.AddEntry(line_fit, "Fit lineare", "l")
leg3.Draw()

c3.Draw()
c3.SaveAs("Immagini/fit_line.png")

Fit status: 0
Parametri stimati:
m = -1.9420256403002156e-07 ± 1.7448704375616856e-06
q = 0.00011856268173460194 ± 0.001069032511382968
Chi2 = 2.2354483640498146
****************************************
Minimizer is Minuit2 / Migrad
Chi2                      =      2.23545
NDf                       =           12
Edm                       =  1.74431e-12
NCalls                    =           41
m                         = -1.94203e-07   +/-   1.74487e-06 
q                         =  0.000118563   +/-   0.00106903  


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: png file Immagini/fit_line.png has been created
